# Steerling-8B: Chunk-to-Concept Attribution

Faithful concept attribution captured **during generation** — each token's attribution
comes from the forward pass where it was committed, using the actual partially-masked context.

For each predicted token, Steerling's concept heads produce per-concept contributions to the
output logit:

$$C(c_i, y_t) = \sigma(\text{logit}_i) \cdot (e_i \cdot W_{y_t})$$

To summarize a chunk, we normalize per-position (so every token votes equally) then average:

$$\text{chunk\_score}(i) = \frac{1}{T} \sum_{t \in \text{chunk}} \frac{C(c_i, y_t)}{\sum_j |C(c_j, y_t)| + |\varepsilon_t|}$$

**Requirements:** Interpretable Steerling model, GPU with >= 18 GB VRAM

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig
from steerling.attribution.concept_attribution import (
    FaithfulConceptAttributor, find_chunks, chunk_attribution,
)

PURPLE = "#675BF2"
PURPLE_LIGHT = "#ceb4fe"
COLOR_MAP = {"known": PURPLE, "discovered": PURPLE_LIGHT}

## 1. Load Model & Concept Labels

In [13]:
model_id = "guidelabs/steerling-8b"

model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")

print(generator)
print(f"Interpretable: {generator.is_interpretable}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

SteerlingGenerator(
  params=8,391,778,304,
  device=cuda,
  interpretable=True,
  diff_block_size=64
)
Interpretable: True


In [ ]:
def find_repo_root(marker="pyproject.toml"):
    for p in [Path().resolve(), *Path().resolve().parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"Could not find repo root (looking for {marker})")

REPO_ROOT = find_repo_root()
concepts_path = REPO_ROOT / "assets" / "concepts" / "concept_labels.parquet"
if not concepts_path.exists():
    concepts_path = None
    print("No concept file found, using fallback labels")
else:
    print(f"Found concept labels at {concepts_path}")

## 2. Create Attributor

The `FaithfulConceptAttributor` wraps the generator and captures attribution
during generation via a step callback.

In [15]:
attributor = FaithfulConceptAttributor(
    generator,
    concepts_path=concepts_path,
    unknown_topk=64,
)

Loaded 33732 concept labels from /fss/aya/home/steerling_public/assets/concepts/known_concepts.csv


## 3. Generate & Attribute

In [16]:
prompt = "AI technology will"
config = GenerationConfig(max_new_tokens=128, steps=128, seed=42)

gen_output, chunk_results = attributor.attribute(prompt, config)

print(f"Generated ({gen_output.generated_tokens} tokens):")
print(gen_output.text)

Generated (128 tokens):
 be able to analyze the tone and emotion of a person's voice, allowing for more personalized and natural interactions.

<|endofchunk|>Enhanced Language Understanding

With advancements in speech recognition and natural language processing, AI virtual assistants will be able to understand and respond to a wider range of user queries. This will enable virtual assistants to provide more accurate and relevant information, enhancing the overall user experience.

Improved Contextual Awareness

As AI technologies improve, they will become better at understanding the context of a conversation or situation. This will allow virtual assistants to provide more appropriate and helpful responses, taking into account the user's previous interactions and preferences.

<|endofchunk|>Real-Time Interaction




## 4. Display Results

Show the top concepts per chunk. Known concepts in purple, discovered in light purple.
Epsilon (residual unexplained by concepts) is shown as a percentage.

In [ ]:
def plot_contributions(entries, title="Concept Contributions", eps_pct=0.0, top_k=20):
    """Horizontal bar chart of top concept contributions."""
    display = sorted(entries, key=lambda e: e["contribution"], reverse=True)[:top_k]

    labels = [e["label"] for e in display]
    values = [e["contribution"] for e in display]
    colors = [COLOR_MAP[e["type"]] for e in display]

    fig, ax = plt.subplots(figsize=(10, max(3, len(labels) * 0.35)))
    y = np.arange(len(labels))
    ax.barh(y, values, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Fraction of total |logit| mass")
    ax.set_title(f"{title}  |  epsilon = {eps_pct:.1f}%", fontsize=11)
    ax.legend(
        handles=[Patch(fc=COLOR_MAP[t], label=t.title()) for t in ["known", "discovered"]],
        loc="lower right", fontsize=8,
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
tokens = gen_output.tokens
prompt_len = gen_output.prompt_tokens
chunks = find_chunks(tokens, generator.tokenizer)
attr = attributor.last_attribution

# Filter to generated content only (clip to after prompt, skip EOS padding)
eos_id = generator.eos_token_id
gen_chunks = []
for s, e in chunks:
    if e <= prompt_len:
        continue
    s = max(s, prompt_len)
    if (tokens[s:e] == eos_id).all():
        continue
    gen_chunks.append((s, e))

for s, e in gen_chunks:
    entries, eps_pct = chunk_attribution(
        attr, s, e, batch=0,
        concept_labels=attributor.labels,
        num_known_concepts=attributor._num_known_concepts,
    )
    text_preview = generator.tokenizer.decode(tokens[s:e].tolist(), skip_special_tokens=True)[:100]

    print(f"\n=== Chunk [{s}, {e}) | epsilon = {eps_pct:.1f}% ===")
    print(f"  Text: {text_preview!r}")
    top = sorted(entries, key=lambda e: e["contribution"], reverse=True)[:10]
    for entry in top:
        print(f"  {entry['type']:<12s} {entry['contribution']:6.2%}  {entry['label']}")

    plot_contributions(
        entries,
        title=f'"{text_preview[:60]}..."',
        eps_pct=eps_pct,
        top_k=20,
    )